# Hybrid Agentic Workflow: ReAct Agent with Free Routing

This notebook demonstrates the implementation of an advanced AI agent using the **PlanReActPlanner** from the Google Agent Development Kit (ADK), integrated with **google.colab.ai** for cost-free query routing.

## 1. Environment Setup and Configuration

We begin by installing the necessary libraries and configuring the execution environment. We combine `google-adk` for complex agent orchestration, `google-genai` for the core model interactions, and the native Colab AI library to act as our free gatekeeper.

In [3]:
# 1.1 Install required libraries
!pip install -q \
    google-adk \
    google-genai \
    pydantic \
    python-dotenv


In [ ]:
# 1.2 Import standard and numerical libraries
import os
import getpass
import numpy as np
from typing import Dict, List, Any, Union

# 1.3 Import Pydantic for data validation
from pydantic import (
    BaseModel,
    Field
)

# 1.4 Import Google Colab AI for free routing
from google.colab import ai

# 1.5 Import GenAI types
from google.genai import types

# 1.6 Import Core ADK components for agent and session management
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.models.google_llm import Gemini

# 1.7 Import Advanced ADK Planners for reasoning loops
from google.adk.planners import PlanReActPlanner

# 1.8 Configure the Gemini API Key
if "GEMINI_API_KEY" not in os.environ:
    print("Please provide your Gemini API Key for the ADK components.")
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Your API Key: ")

## 2. Intent Classification and Guardrails (Free Routing)

This section implements the initial routing logic. We use the native `google.colab.ai` text generation capability to act as a free gatekeeper. By defining explicit constraints in the prompt, the system evaluates the user query and routes it to the correct processing engine, preventing the paid ADK ReAct agent from wasting quota on out-of-scope requests.

In [4]:
def classify_intent(query: str) -> str:
    """
    Classify the user intent using free Colab AI to enforce scope constraints.

    Parameters
    ----------
    query : str
        The raw text input provided by the user.

    Returns
    -------
    str
        The classification label: 'MATH', 'THEORY', or 'OUT_OF_SCOPE'.
    """
    routing_prompt = (
        f"Analyze the following user query: '{query}'. "
        "Apply the following strict rules:\n"
        "1. If the query asks for calculations, mean, standard deviation, or provides "
        "a numerical dataset, respond ONLY with 'MATH'.\n"
        "2. If it is a theoretical question related to data science, statistics, "
        "or mathematics, respond ONLY with 'THEORY'.\n"
        "3. If the query is about general knowledge, history, politics, or anything "
        "unrelated to data analysis, respond ONLY with 'OUT_OF_SCOPE'."
    )

    # Leverage the built-in free model to classify the text
    classification = ai.generate_text(routing_prompt)

    return classification.strip().upper()

## 3. Tool Definition: Validated and Resilient Analysis

We use **Pydantic Models** to define a strict contract for our tools. Additionally, we implement **Graceful Error Handling**. Instead of allowing Python to raise unhandled exceptions, our tools return descriptive error messages. This allows the **PlanReActPlanner** to understand what went wrong and communicate the issue back to the user without crashing the workflow.

In [5]:
# 3.1 Schema for Statistics calculation
class StatisticsInput(BaseModel):
    """Schema for the calculate_statistics tool."""

    numbers: List[float] = Field(
        description="A list of numerical values to process."
    )


# 3.2 Tool for calculating mean and standard deviation
def calculate_statistics(
    args: StatisticsInput
) -> Dict[str, Union[float, str]]:
    """
    Calculate the mean and standard deviation of a list of numbers.

    Parameters
    ----------
    args : StatisticsInput
        The validated input containing the list of numbers.

    Returns
    -------
    dict[str, Union[float, str]]
        A dictionary containing the calculated statistics or an error message.
    """
    data = np.array(args.numbers)

    # Error Handling: Check if the list is empty to prevent NumPy warnings/errors
    if data.size == 0:
        return {
            "error": "The provided dataset is empty. Please provide numbers."
        }

    return {
        "mean": float(
            np.mean(data)
        ),
        "standard_deviation": float(
            np.std(data)
        )
    }


# 3.3 Schema for filtering outliers
class FilterInput(BaseModel):
    """Schema for the filter_outliers tool."""

    numbers: List[float] = Field(
        description="The original list of numbers to be filtered."
    )
    mean: float = Field(
        description="The arithmetic mean used as a reference point."
    )
    std_dev: float = Field(
        description="The standard deviation used to define the spread."
    )
    threshold: float = Field(
        default=2.0,
        description="The number of standard deviations for the outlier limit."
    )


# 3.4 Tool for filtering out values beyond a statistical threshold
def filter_outliers(
    args: FilterInput
) -> Union[List[float], Dict[str, str]]:
    """
    Remove numbers that are more than a certain number of standard deviations
    away from the mean.

    Parameters
    ----------
    args : FilterInput
        The validated input containing the data and filtering parameters.

    Returns
    -------
    list of float or dict[str, str]
        A filtered list or an error message if the input is invalid.
    """
    # Error Handling: Prevent division by zero or invalid std_dev logic
    if args.std_dev <= 0:
        return {
            "error": "Standard deviation must be greater than zero to filter."
        }

    return [
        x for x in args.numbers
        if abs(x - args.mean) <= (args.threshold * args.std_dev)
    ]

## 4. Agent Configuration with Reasoning and Guardrails

In this section, we define the core intelligence of our assistant. We use the `PlanReActPlanner` to enable multi-step logic. We also implement defense-in-depth **Guardrails** within the system instructions to ensure the agent only operates within its specialized domain of numerical analysis, even if a query bypasses the initial router.

The agent is programmed to:
1. Validate the input as a numerical dataset.
2. Follow a Thought-Action-Observation cycle to clean and analyze data.
3. Understand and explain tool errors to the user instead of failing blindly.

In [6]:
# 4.1 Initialize the strategic planner
planner = PlanReActPlanner()

# 4.2 Define the model with automated retry logic
# initial_delay=1 means it waits 1 second before the first retry
# attempts=3 means it will try up to 3 times before giving up
gemini_model = Gemini(
    model="gemini-flash-latest",
    retry_options=types.HttpRetryOptions(
        initial_delay=1,
        attempts=3
    )
)

# 4.3 Define the specialized agent with built-in constraints and error awareness
statistical_agent = LlmAgent(
    model=gemini_model,
    name="specialized_data_analyst",
    description="Authorized only for statistical analysis and outlier detection.",
    instruction="""You are a specialized Data Science Assistant.

    **Scope of Action**:
    - You ONLY process numerical datasets for statistical summaries.
    - You ONLY use the provided tools (calculate_statistics, filter_outliers).
    - Strictly refuse to answer general knowledge, history, or coding questions.

    **Execution Protocol**:
    1. Analyze if the user query contains a numerical dataset.
    2. If the request is outside your scope, inform the user immediately.
    3. If a tool returns an 'error' message, explain the issue clearly to the
       user instead of proceeding with the plan.
    4. If valid, calculate initial statistics, filter outliers (>2 std dev),
       and provide final cleaned results.
    5. Present 'before' and 'after' metrics using a professional and direct tone.""",
    tools=[
        calculate_statistics,
        filter_outliers
    ],
    planner=planner
)

## 5. Session Management and Hybrid Execution Framework

In this final section, we manage the conversation state using `InMemorySessionService` and initialize the `Runner`. 

We then define the `execute_hybrid_analysis` orchestrator. This function serves as the central hub:
1. It calls the free `colab.ai` router to classify the query.
2. It blocks out-of-scope requests immediately via Python (Zero API cost).
3. It handles theoretical questions using the free Colab AI text generator.
4. It delegates numerical datasets to the ADK `PlanReActPlanner`, printing the detailed Thought-Action-Observation reasoning trace.

In [7]:
# 5.1 Configuration Variables
APP_NAME = "advanced_statistical_analysis"
USER_ID = "colab_user"
SESSION_ID = "session_react_hybrid_01"

# 5.2 Initialize the service that manages conversation state in memory
session_service = InMemorySessionService()

# 5.3 Create a specific session for this job
# In Colab, we use 'await' directly at the top level
await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID
)

# 5.4 Configure the Runner, linking the agent to the session service
executor = Runner(
    agent=statistical_agent,
    app_name=APP_NAME,
    session_service=session_service
)

In [8]:
async def execute_hybrid_analysis(query: str) -> None:
    """
    Execute the hybrid workflow: route via free AI, then process if valid.
    Prints intermediate reasoning steps if routed to the ReAct agent.

    Parameters
    ----------
    query : str
        The natural language query provided by the user.

    Returns
    -------
    None
    """
    print("=" * 60)
    print(f"User Query: {query}\n")
    print("--- System Routing ---")

    # Step A: Classify the intent using our free router
    intent = classify_intent(query)
    print(f"Intent Classified as: [{intent}]\n")

    # Step B: Logic Execution based on Intent
    if "OUT_OF_SCOPE" in intent:
        # Constraint applied locally without spending API quota
        print("[System] I am a specialized Data Science Assistant. "
              "I am not authorized to answer general knowledge, history, "
              "or unrelated questions.\n")
        return

    elif "THEORY" in intent:
        # Route to free colab.ai engine
        print("--- Free AI Response (Theory) ---\n")
        stream = ai.generate_text(query, stream=True)
        for chunk in stream:
            print(chunk, end='')
        print("\n")

    else:
        # Route to Paid ADK Agent (MATH)
        print("--- Reasoning Trace (Paid ADK Agent) ---")
        message = types.Content(
            role="user",
            parts=[
                types.Part(text=query)
            ]
        )

        # Start asynchronous execution through the Runner
        async for event in executor.run_async(
            user_id=USER_ID,
            session_id=SESSION_ID,
            new_message=message
        ):
            # Parse the reasoning trace accurately as defined in the original code
            if event.content and event.content.parts:
                for part in event.content.parts:
                    if part.function_call:
                        print(f"\n[Action] Calling tool: {part.function_call.name}")

                    elif part.text:
                        text_content = part.text.strip()
                        if "/*REASONING*/" in text_content:
                            print(f"\n[Thought]\n{text_content.replace('/*REASONING*/', '').strip()}")
                        elif "/*FINAL_ANSWER*/" in text_content:
                            print(f"\n[Final Answer]\n{text_content.replace('/*FINAL_ANSWER*/', '').strip()}")
                        else:
                            print(f"\n{text_content}")
        print("\n")

In [ ]:
# Test 1: Out of scope (Will use ZERO API quota and block the request)
await execute_hybrid_analysis(
    "Who is the current president of Brazil?"
)

In [ ]:
# Test 2: Theoretical concept (Will use ZERO API quota and stream response)
await execute_hybrid_analysis(
    "Can you explain what an outlier is in statistics?"
)

In [ ]:
# Test 3: Complex Mathematics (Will use Paid API and ReAct loops)
await execute_hybrid_analysis(
    "Analyze this: [10, 12, 11, 15, 100, 13, 14]. Clean it and give me the mean."
)